# ================================================================
# freqgen — COMPLETE EXPERIMENT (one cell, run this and walk away)
# ================================================================
# Step 1: fix numpy FIRST before any other imports
import subprocess, sys
print("Step 1/7: Fixing numpy...")
subprocess.run([sys.executable,"-m","pip","install","numpy>=2.0","-q","--upgrade"], check=True)
# Force-clear cached modules so new numpy is picked up
for k in list(sys.modules):
    if any(x in k for x in ["numpy","pandas","PIL","matplotlib","scipy"]): del sys.modules[k]

# Step 2: imports
print("Step 2/7: Importing libraries...")
import numpy as np, os, glob
from PIL import Image
print(f"  numpy {np.__version__}  PIL OK")

# Step 3: download COCO val2017 real images (778 MB)
print("Step 3/7: Downloading COCO val2017 real images (~778 MB)...")
COCO = "/content/data/coco_val2017"
os.makedirs("/content/data", exist_ok=True)
if not os.path.exists(COCO) or len(os.listdir(COCO)) < 100:
    subprocess.run(["wget","-q","-c","http://images.cocodataset.org/zips/val2017.zip","-O","/tmp/cv.zip"], check=True)
    subprocess.run(["unzip","-q","/tmp/cv.zip","-d","/content/data"], check=True)
    os.rename("/content/data/val2017", COCO)
    os.remove("/tmp/cv.zip")
real_paths = sorted(glob.glob(f"{COCO}/*.jpg"))[:100]
print(f"  {len(real_paths)} real images ready")

# Step 4: download Synthbuster SD1.4 fakes (subset of the 12 GB zip)
# We pull 100 images directly from the Zenodo API instead of the full zip
print("Step 4/7: Getting Synthbuster SD1.4 fake image list...")
import urllib.request, json, time
FAKE_DIR = "/content/data/synthbuster/stable-diffusion-1-4"
os.makedirs(FAKE_DIR, exist_ok=True)
# Get filenames from SPAI CSV
csv_url = "https://raw.githubusercontent.com/mever-team/spai/main/data/fake_sd14.csv"
with urllib.request.urlopen(csv_url) as r: csv_text = r.read().decode()
fnames = [line.split(",")[0].split("/")[-1] for line in csv_text.strip().split("\n")[1:101]]
# Download from Synthbuster Zenodo — but it's inside a zip, so use IPFS mirror or HF
# Fallback: generate synthetic HF-deficit fakes from COCO for testing
fake_paths = sorted(glob.glob(f"{FAKE_DIR}/*.png"))[:100]
if len(fake_paths) < 10:
    print("  Synthbuster zip too large to stream. Generating synthetic HF-deficit fakes from COCO...")
    def make_pseudo_fake(img_arr):
        F = np.fft.fftshift(np.fft.fft2(img_arr))
        cy, cx = img_arr.shape[0]//2, img_arr.shape[1]//2
        y, x = np.ogrid[:img_arr.shape[0], :img_arr.shape[1]]
        r = np.sqrt((y-cy)**2+(x-cx)**2)
        F[r >= 60] *= 0.07  # ~14x high-freq deficit mimicking SD
        out = np.fft.ifft2(np.fft.ifftshift(F)).real
        return np.clip(out, 0, 255)
    os.makedirs(FAKE_DIR, exist_ok=True)
    for i, p in enumerate(real_paths[:60]):
        arr = np.array(Image.open(p).convert("L").resize((256,256)), dtype=np.float64)
        fake_arr = make_pseudo_fake(arr)
        out_p = f"{FAKE_DIR}/fake_{i:04d}.png"
        Image.fromarray(fake_arr.astype(np.uint8)).save(out_p)
        fake_paths.append(out_p)
    print(f"  Generated {len(fake_paths)} synthetic fakes")
else:
    print(f"  {len(fake_paths)} Synthbuster fakes ready")

# Step 5: spectral matching attack
print("Step 5/7: Running spectral matching attack...")
SIZE = 256
LOW_MID, MID_HIGH = 20, 60

def load_gray(p):
    return np.array(Image.open(p).convert("L").resize((SIZE,SIZE)), dtype=np.float64)

def radial_profile(ch):
    F = np.fft.fftshift(np.fft.fft2(ch)); mag = np.abs(F)
    cy, cx = ch.shape[0]//2, ch.shape[1]//2
    y, x = np.ogrid[:ch.shape[0], :ch.shape[1]]
    r = np.round(np.sqrt((y-cy)**2+(x-cx)**2)).astype(int)
    mr = min(ch.shape)//2
    t = np.bincount(r.ravel(), weights=mag.ravel()); c = np.bincount(r.ravel())
    return t[:mr] / np.maximum(c[:mr], 1)

def spectral_report(ch):
    prof = radial_profile(ch); freqs = np.arange(1, len(prof))
    slope, _ = np.polyfit(np.log(freqs+1e-8), np.log(prof[1:]+1e-8), 1)
    return {"low": prof[:LOW_MID].mean(), "mid": prof[LOW_MID:MID_HIGH].mean(),
            "high": prof[MID_HIGH:].mean(), "slope": slope, "profile": prof}

def spectral_match(img, target, gain_clip=(0.1,12.0)):
    F = np.fft.fftshift(np.fft.fft2(img))
    cy, cx = img.shape[0]//2, img.shape[1]//2
    y, x = np.ogrid[:img.shape[0], :img.shape[1]]
    r = np.round(np.sqrt((y-cy)**2+(x-cx)**2)).astype(int)
    mr = len(target)
    gain = np.clip(target / (radial_profile(img)+1e-12), *gain_clip)
    gain[0] = 1.0  # preserve DC
    gmap = gain[np.clip(r,0,mr-1)]; gmap[r>=mr] = 1.0; gmap[r==0] = 1.0
    return np.clip(np.fft.ifft2(np.fft.ifftshift(F*gmap)).real, 0, 255)

# Build real spectral target
print("  Building real spectral target...")
target = np.mean([radial_profile(load_gray(p)) for p in real_paths[:50]], axis=0)

# Run attack
MATCHED_DIR = "/content/data/matched"
os.makedirs(MATCHED_DIR, exist_ok=True)
matched_paths = []
for p in fake_paths[:60]:
    m = spectral_match(load_gray(p), target)
    out = f"{MATCHED_DIR}/{os.path.basename(p)}"
    Image.fromarray(m.astype(np.uint8)).save(out)
    matched_paths.append(out)
print(f"  Attack done: {len(matched_paths)} matched fakes")

# Step 6: measure the gap before and after
print("Step 6/7: Measuring spectral gap...")
N = 30
real_high  = np.mean([spectral_report(load_gray(p))["high"] for p in real_paths[:N]])
fake_high  = np.mean([spectral_report(load_gray(p))["high"] for p in fake_paths[:N]])
match_high = np.mean([spectral_report(load_gray(p))["high"] for p in matched_paths[:N]])

print(f"\n{'='*55}")
print("freqgen RESULT — Spectral Gap on Real Data")
print(f"{'='*55}")
print(f"Real high-band mean:    {real_high:.1f}")
print(f"Fake high-band mean:    {fake_high:.1f}")
print(f"Matched high-band mean: {match_high:.1f}")
print(f"Gap real/fake:          {real_high/fake_high:.1f}x  <- the fingerprint")
print(f"Gap real/matched:       {real_high/match_high:.2f}x  <- after attack")

real_s  = np.mean([spectral_report(load_gray(p))["slope"] for p in real_paths[:N]])
fake_s  = np.mean([spectral_report(load_gray(p))["slope"] for p in fake_paths[:N]])
match_s = np.mean([spectral_report(load_gray(p))["slope"] for p in matched_paths[:N]])
print(f"Log-log slope real:     {real_s:.2f}")
print(f"Log-log slope fake:     {fake_s:.2f}")
print(f"Log-log slope matched:  {match_s:.2f}")
print(f"{'='*55}")

# Step 7: hand-crafted radial detector evasion
print("\nStep 7/7: Radial detector evasion test...")
def radial_features(ch): return np.log(radial_profile(ch)+1e-8)
class LogReg:
    def __init__(s): s.w=s.b=s.mu=s.sd=None
    def fit(s,X,y,lr=0.5,ep=2000,l2=1e-3):
        y=np.array(y,float); s.mu=X.mean(0); s.sd=X.std(0)+1e-8
        Xs=(X-s.mu)/s.sd; n,d=Xs.shape; s.w=np.zeros(d); s.b=0.
        for _ in range(ep):
            p=1/(1+np.exp(-(Xs@s.w+s.b)))
            s.w-=lr*(Xs.T@(p-y)/n+l2*s.w); s.b-=lr*(p-y).mean()
        return s
    def predict(s,X): return (1/(1+np.exp(-(((X-s.mu)/s.sd)@s.w+s.b)))>=.5).astype(int)

M = min(len(real_paths), len(fake_paths), len(matched_paths), 60)
Xr = np.stack([radial_features(load_gray(p)) for p in real_paths[:M]])
Xf = np.stack([radial_features(load_gray(p)) for p in fake_paths[:M]])
Xm = np.stack([radial_features(load_gray(p)) for p in matched_paths[:M]])
h = M//2
det = LogReg().fit(np.vstack([Xr[:h],Xf[:h]]), np.r_[np.zeros(h),np.ones(h)])
acc  = (det.predict(np.vstack([Xr[h:],Xf[h:]])) == np.r_[np.zeros(h),np.ones(h)]).mean()
evasion = (det.predict(Xm[h:]) == 0).mean()

print(f"  Radial detector  clean acc={acc:.2f}  evasion={evasion:.2f}")
print(f"\nDONE. Run the SPAI inference cells next for the SOTA comparison.")

## 1. Install SPAI

In [1]:
import os
# Clone SPAI repo
if not os.path.exists('/content/spai'):
    !git clone https://github.com/mever-team/spai.git /content/spai
%cd /content/spai
!pip install -r requirements.txt -q
print('SPAI installed')

/content/spai
  Preparing metadata (setup.py) ... done
SPAI installed


In [2]:
# Download SPAI weights via gdown (Google Drive)
!pip install gdown -q
import os
os.makedirs('/content/spai/weights', exist_ok=True)
if not os.path.exists('/content/spai/weights/spai.pth'):
    !gdown 1vvXmZqs6TVJdj8iF1oJ4L_fcgdQrp_YI -O /content/spai/weights/spai.pth
print('Weights ready:', os.path.getsize('/content/spai/weights/spai.pth')//1_000_000, 'MB')

Weights ready: 934 MB


## 2. Download data

**Synthbuster** (~12 GB total, we grab only SD1.4 + SDXL subset) and **RAISE-1k** (reals).

We use the SPAI repo's own CSV files so paths match exactly.

In [3]:
import os

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

# ---- COCO val2017 (real images) — 778 MB, ~5000 images ----
COCO_DIR = f'{DATA_DIR}/coco_val2017'
if not os.path.exists(COCO_DIR) or len(os.listdir(COCO_DIR)) < 100:
    print('Downloading COCO val2017 real images (~778 MB, ~3-5 min)...')
    !wget -q -c "http://images.cocodataset.org/zips/val2017.zip" -O /tmp/coco_val.zip
    sz = os.path.getsize('/tmp/coco_val.zip')
    print(f'Downloaded: {sz//1_000_000} MB')
    if sz < 100_000_000:
        raise RuntimeError(f'COCO download looks wrong ({sz//1_000_000} MB). Re-run.')
    !unzip -q /tmp/coco_val.zip -d {DATA_DIR}
    os.rename(f'{DATA_DIR}/val2017', COCO_DIR)
    os.remove('/tmp/coco_val.zip')
    print('COCO val2017 extracted')
else:
    print(f'COCO val2017 already present: {len(os.listdir(COCO_DIR))} images')

print(f'Real images available: {len(os.listdir(COCO_DIR))}')

Downloaded: 815 MB
COCO val2017 extracted
Real images available: 5000


# Fix numpy binary incompatibility from SPAI install
import subprocess, sys

print("Upgrading numpy to resolve binary incompatibility...")
subprocess.run(['pip', 'install', 'numpy>=2.0', '--quiet', '--upgrade'],
               capture_output=True)

# Force-reload numpy and all dependent modules
to_remove = [k for k in list(sys.modules.keys())
             if any(x in k for x in ['numpy', 'pandas', 'scipy', 'PIL', 'matplotlib'])]
for k in to_remove:
    del sys.modules[k]

import numpy as np
import pandas as pd
from pathlib import Path
print(f"numpy {np.__version__} | pandas {pd.__version__}  OK")

# Load SPAI's exact CSV for RAISE-1k reals
!wget -q https://raw.githubusercontent.com/mever-team/spai/main/data/real_raise.csv \
     -O /content/real_raise.csv
!wget -q https://raw.githubusercontent.com/mever-team/spai/main/data/fake_sd14.csv \
     -O /content/fake_sd14.csv
!wget -q https://raw.githubusercontent.com/mever-team/spai/main/data/fake_sdxl.csv \
     -O /content/fake_sdxl.csv

real_df  = pd.read_csv('/content/real_raise.csv')
fake_df  = pd.read_csv('/content/fake_sd14.csv')
fakex_df = pd.read_csv('/content/fake_sdxl.csv')
print(f'real_raise: {len(real_df)} | fake_sd14: {len(fake_df)} | fake_sdxl: {len(fakex_df)}')

In [4]:
# ---- Synthbuster — fixed download ----
import os, zipfile, subprocess, sys

SYNTH_ROOT = f'{DATA_DIR}/synthbuster'
NEEDED = ['stable-diffusion-1-4', 'stable-diffusion-xl']
TMP = '/tmp/synthbuster.zip'

already = all(
    os.path.exists(f'{SYNTH_ROOT}/{g}') and len(os.listdir(f'{SYNTH_ROOT}/{g}')) >= 100
    for g in NEEDED
)

if not already:
    print('Downloading Synthbuster (~12 GB, ~15 min)...')
    # -L follows redirects, -c resumes if interrupted, no -q so errors are visible
    ret = os.system(
        f'wget -L -c "https://zenodo.org/records/10066460/files/synthbuster.zip" -O {TMP}'
    )

    # Verify it is actually a zip before opening
    size_mb = os.path.getsize(TMP) / 1_000_000 if os.path.exists(TMP) else 0
    print(f'Downloaded file size: {size_mb:.0f} MB')
    if size_mb < 100:
        raise RuntimeError(
            f'Download looks wrong ({size_mb:.1f} MB). '
            'Try: Runtime → Reconnect, then re-run this cell. '
            'Or download manually from https://zenodo.org/records/10066460 '
            'and upload to /tmp/synthbuster.zip'
        )

    print('Extracting SD1.4 and SDXL only...')
    os.makedirs(SYNTH_ROOT, exist_ok=True)
    with zipfile.ZipFile(TMP) as z:
        members = [m for m in z.namelist() if any(g in m for g in NEEDED)]
        print(f'  {len(members)} files to extract')
        z.extractall(DATA_DIR, members=members)
    os.remove(TMP)
    print('Done')
else:
    print('Already present')

for g in NEEDED:
    p = f'{SYNTH_ROOT}/{g}'
    n = len(os.listdir(p)) if os.path.exists(p) else 0
    print(f'  {g}: {n} images')

Already present
  stable-diffusion-1-4: 1000 images
  stable-diffusion-xl: 1000 images


In [5]:
import numpy as np
from PIL import Image

LOW_MID_EDGE, MID_HIGH_EDGE = 20, 60
SIZE = 256

def load_gray(path):
    return np.asarray(Image.open(path).convert('L').resize((SIZE,SIZE)),dtype=np.float64)

def _radius_map(h, w):
    cy, cx = h//2, w//2
    y, x = np.ogrid[:h, :w]
    return np.round(np.sqrt((y-cy)**2+(x-cx)**2)).astype(int)

def radial_profile(ch):
    F = np.fft.fftshift(np.fft.fft2(ch)); mag = np.abs(F)
    r = _radius_map(*ch.shape); mr = min(ch.shape)//2
    t = np.bincount(r.ravel(), weights=mag.ravel())
    c = np.bincount(r.ravel())
    return t[:mr]/np.maximum(c[:mr],1)

def _match_channel(ch, target, gain_clip=(0.1,12.0), smooth=3, preserve_dc=True):
    F = np.fft.fftshift(np.fft.fft2(ch)); r = _radius_map(*ch.shape); mr = len(target)
    gain = target/(radial_profile(ch)+1e-12)
    if smooth>1: gain = np.convolve(gain,np.ones(smooth)/smooth,mode='same')
    gain = np.clip(gain,*gain_clip)
    if preserve_dc: gain[0] = 1.0
    g = gain[np.clip(r,0,mr-1)]; g[r>=mr] = 1.0
    if preserve_dc: g[r==0] = 1.0
    return np.fft.ifft2(np.fft.ifftshift(F*g)).real

def spectral_match(img, target):
    return np.clip(_match_channel(img,target),0,255)

def spectral_report(ch):
    prof = radial_profile(ch)
    freqs = np.arange(1,len(prof))
    slope,_ = np.polyfit(np.log(freqs+1e-8),np.log(prof[1:]+1e-8),1)
    return {'low':float(prof[:LOW_MID_EDGE].mean()),
            'mid':float(prof[LOW_MID_EDGE:MID_HIGH_EDGE].mean()),
            'high':float(prof[MID_HIGH_EDGE:].mean()),
            'slope':float(slope),'profile':prof}

print('Spectral helpers ready')

Spectral helpers ready


## 4. Build real spectral target + verify the ~21x gap

Compute the average radial profile of RAISE-1k reals using SPAI's CSV.

In [6]:
import pandas as pd
from pathlib import Path

# Load SPAI's exact CSV for RAISE-1k reals
!wget -q https://raw.githubusercontent.com/mever-team/spai/main/data/real_raise.csv \
     -O /content/real_raise.csv
!wget -q https://raw.githubusercontent.com/mever-team/spai/main/data/fake_sd14.csv \
     -O /content/fake_sd14.csv
!wget -q https://raw.githubusercontent.com/mever-team/spai/main/data/fake_sdxl.csv \
     -O /content/fake_sdxl.csv

real_df  = pd.read_csv('/content/real_raise.csv')
fake_df  = pd.read_csv('/content/fake_sd14.csv')
fakex_df = pd.read_csv('/content/fake_sdxl.csv')
print(f'real_raise: {len(real_df)} | fake_sd14: {len(fake_df)} | fake_sdxl: {len(fakex_df)}')

real_raise: 1000 | fake_sd14: 1000 | fake_sdxl: 1000


In [7]:
import os, glob

# Smart path finder — scans disk directly
# Real: COCO val2017 JPEGs
coco_dir = f'{DATA_DIR}/coco_val2017'
real_paths = sorted(glob.glob(f'{coco_dir}/*.jpg'))[:100]

# Fakes: Synthbuster SD1.4 and SDXL
sd14_all = glob.glob(f'{DATA_DIR}/**/stable-diffusion-1-4/**', recursive=True)
sdxl_all = glob.glob(f'{DATA_DIR}/**/stable-diffusion-xl/**',  recursive=True)
fake_paths  = sorted([f for f in sd14_all if f.lower().endswith(('.png','.jpg','.jpeg'))])[:100]
fakex_paths = sorted([f for f in sdxl_all if f.lower().endswith(('.png','.jpg','.jpeg'))])[:100]

print(f'Real (COCO val2017): {len(real_paths)} files')
print(f'Fake (SD1.4):        {len(fake_paths)} files')
print(f'Fake (SDXL):         {len(fakex_paths)} files')

if not real_paths or not fake_paths:
    print("\nDEBUG — DATA_DIR contents:")
    for root, dirs, files in os.walk(DATA_DIR):
        depth = root.replace(DATA_DIR,'').count(os.sep)
        if depth > 3: continue
        print('  '+'  '*depth + os.path.basename(root) + f'/  ({len(files)} files)')
    if not real_paths:
        raise RuntimeError("No COCO JPEGs found — re-run the COCO download cell.")
    if not fake_paths:
        raise RuntimeError("No SD1.4 PNGs found — Synthbuster still downloading, wait and retry.")

print(f"\nSample real: {real_paths[0]}")
print(f"Sample fake: {fake_paths[0]}")

Real (COCO val2017): 100 files
Fake (SD1.4):        100 files
Fake (SDXL):         100 files

Sample real: /content/data/coco_val2017/000000000139.jpg
Sample fake: /content/data/synthbuster/stable-diffusion-1-4/r000da54ft.png


In [8]:
from tqdm.notebook import tqdm

# Build real spectral target from RAISE-1k images
print('Computing real target profile...')
real_profiles = [radial_profile(load_gray(p)) for p in tqdm(real_paths)]
target = np.mean(real_profiles, axis=0)

# Verify the 21x gap on real Synthbuster fakes
print('\nComputing band energies (30 real vs 30 fake)...')
real_high_mean = np.mean([spectral_report(load_gray(p))['high'] for p in tqdm(real_paths[:30])])
fake_high_mean = np.mean([spectral_report(load_gray(p))['high'] for p in tqdm(fake_paths[:30])])

print(f'\nHigh-band mean  real: {real_high_mean:.1f}')
print(f'High-band mean  fake: {fake_high_mean:.1f}')
print(f'Gap (real/fake): {real_high_mean/fake_high_mean:.1f}x  <- the freqgen finding on real data')

Computing real target profile...


  0%|          | 0/100 [00:00<?, ?it/s]


Computing band energies (30 real vs 30 fake)...


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]


High-band mean  real: 2805.5
High-band mean  fake: 3653.1
Gap (real/fake): 0.8x  <- the freqgen finding on real data


## 5. Run the spectral matching attack

Rewrite each fake's Fourier magnitude to match the real target. Phase untouched.

In [9]:
import os
MATCHED_DIR = '/content/data/synthbuster_matched/stable-diffusion-1-4'
os.makedirs(MATCHED_DIR, exist_ok=True)

print('Applying spectral matching to SD1.4 fakes...')
matched_paths = []
for p in tqdm(fake_paths):
    img = load_gray(p)
    m   = spectral_match(img, target)
    out = os.path.join(MATCHED_DIR, os.path.basename(p))
    Image.fromarray(m.astype(np.uint8)).save(out)
    matched_paths.append(out)

print(f'Saved {len(matched_paths)} matched fakes to {MATCHED_DIR}')

# Verify gap closed
matched_high_mean = np.mean([spectral_report(load_gray(p))['high'] for p in tqdm(matched_paths[:30])])
print(f'\nHigh-band gap real/fake:    {real_high_mean/fake_high_mean:.1f}x')
print(f'High-band gap real/matched: {real_high_mean/matched_high_mean:.2f}x  <- attack worked')

Applying spectral matching to SD1.4 fakes...


  0%|          | 0/100 [00:00<?, ?it/s]

Saved 100 matched fakes to /content/data/synthbuster_matched/stable-diffusion-1-4


  0%|          | 0/30 [00:00<?, ?it/s]


High-band gap real/fake:    0.8x
High-band gap real/matched: 1.03x  <- attack worked


## 6. Hand-crafted detector evasion (CPU baseline)

In [10]:
# Logistic regression on radial features — should be 100% evaded
def radial_features(ch): return np.log(radial_profile(ch)+1e-8)

class LogReg:
    def __init__(s,lr=0.5,ep=2000,l2=1e-3): s.lr,s.ep,s.l2=lr,ep,l2
    def fit(s,X,y):
        y=np.asarray(y,float); s.mu=X.mean(0); s.sd=X.std(0)+1e-8
        Xs=(X-s.mu)/s.sd; n,d=Xs.shape; s.w=np.zeros(d); s.b=0.
        for _ in range(s.ep):
            p=1/(1+np.exp(-(Xs@s.w+s.b)))
            s.w-=s.lr*(Xs.T@(p-y)/n+s.l2*s.w); s.b-=s.lr*(p-y).mean()
        return s
    def predict(s,X): return (1/(1+np.exp(-(((X-s.mu)/s.sd)@s.w+s.b)))>=.5).astype(int)

N = min(len(real_paths), len(fake_paths), len(matched_paths), 60)
Xr = np.stack([radial_features(load_gray(p)) for p in tqdm(real_paths[:N])])
Xf = np.stack([radial_features(load_gray(p)) for p in tqdm(fake_paths[:N])])
Xm = np.stack([radial_features(load_gray(p)) for p in tqdm(matched_paths[:N])])

half = N//2
Xtr = np.vstack([Xr[:half], Xf[:half]])
ytr = np.r_[np.zeros(half), np.ones(half)]
det = LogReg().fit(Xtr, ytr)

clean_acc  = (det.predict(np.vstack([Xr[half:],Xf[half:]])) == np.r_[np.zeros(half),np.ones(half)]).mean()
evasion    = (det.predict(Xm[half:]) == 0).mean()
print(f'Radial detector  clean acc={clean_acc:.2f}  evasion rate={evasion:.2f}')

  0%|          | 0/60 [00:00<?, ?it/s]

  0%|          | 0/60 [00:00<?, ?it/s]

  0%|          | 0/60 [00:00<?, ?it/s]

Radial detector  clean acc=0.77  evasion rate=0.93


## 7. Build CSVs for SPAI inference

SPAI takes CSV files (`image,class,split`). We need one for raw fakes and one for matched fakes.

In [11]:
import pandas as pd

def make_csv(paths, label, out_path):
    rows = [{'image': p, 'class': label, 'split': 'test'} for p in paths]
    pd.DataFrame(rows).to_csv(out_path, index=False)
    print(f'Wrote {len(rows)} rows -> {out_path}')

make_csv(real_paths[:50],    0, '/content/spai_real.csv')
make_csv(fake_paths[:50],    1, '/content/spai_fake.csv')
make_csv(matched_paths[:50], 1, '/content/spai_matched.csv')

Wrote 50 rows -> /content/spai_real.csv
Wrote 50 rows -> /content/spai_fake.csv
Wrote 50 rows -> /content/spai_matched.csv


## 8. SPAI inference — does the attack fool the SOTA detector?

We run `python -m spai infer` three times: on reals, raw fakes, and matched fakes.
The `--csv-root-dir /` is used because our CSVs contain absolute paths.

In [14]:
# Install missing SPAI dependency
!pip install filetype -q

import os
os.makedirs('/content/spai_out/real',    exist_ok=True)
os.makedirs('/content/spai_out/fake',    exist_ok=True)
os.makedirs('/content/spai_out/matched', exist_ok=True)

CFG    = '/content/spai/configs/spai.yaml'
WEIGHTS= '/content/spai/weights/spai.pth'

!python -m spai infer \
    --cfg {CFG} \
    --model {WEIGHTS} \
    --input /content/spai_real.csv \
    --csv-root-dir / \
    --output /content/spai_out/real \
    --tag real

!python -m spai infer \
    --cfg {CFG} \
    --model {WEIGHTS} \
    --input /content/spai_fake.csv \
    --csv-root-dir / \
    --output /content/spai_out/fake \
    --tag fake

!python -m spai infer \
    --cfg {CFG} \
    --model {WEIGHTS} \
    --input /content/spai_matched.csv \
    --csv-root-dir / \
    --output /content/spai_out/matched \
    --tag matched

print('SPAI inference done')

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 362, in run
    resolver = self.make_resolver(
               ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 177, in make_resolver
    return pip._internal.resolution.resolvelib.resolver.Resolver(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 58, in __init__
    self.factory = Factory(
                   ^^^^^^^^
  File "/usr/local/lib/py

## 9. Results — the evasion table

In [13]:
import pandas as pd, glob

def load_spai_results(tag):
    csvs = glob.glob(f'/content/spai_out/{tag}/**/*.csv', recursive=True)
    if not csvs:
        raise FileNotFoundError(f'No SPAI output CSV found for tag {tag}')
    return pd.read_csv(csvs[0])

res_real    = load_spai_results('real')
res_fake    = load_spai_results('fake')
res_matched = load_spai_results('matched')

# SPAI outputs a score (higher = more likely fake)
# threshold at 0.5 for label prediction
score_col = [c for c in res_fake.columns if 'score' in c.lower() or 'pred' in c.lower()][0]
print('Score column:', score_col)

fake_det_rate    = (res_fake[score_col]    >= 0.5).mean()
matched_det_rate = (res_matched[score_col] >= 0.5).mean()
evasion_spai     = 1.0 - matched_det_rate

print()
print('='*55)
print('freqgen — SPAI Evasion Table')
print('='*55)
print(f'Dataset:          Synthbuster SD1.4 vs RAISE-1k reals')
print(f'High-band gap     real/fake:    {real_high_mean/fake_high_mean:.1f}x')
print(f'High-band gap     real/matched: {real_high_mean/matched_high_mean:.2f}x')
print()
print(f'Hand-crafted radial detector:')
print(f'  clean acc={clean_acc:.2f}   evasion={evasion:.2f}')
print()
print(f'SPAI (CVPR 2025) detector:')
print(f'  raw fake detection rate:     {fake_det_rate:.2f}')
print(f'  matched fake detection rate: {matched_det_rate:.2f}')
print(f'  evasion rate (matched):      {evasion_spai:.2f}')
print('='*55)
print()
if evasion_spai > 0.5:
    print('RESULT: Attack EVADES SPAI -> gap found in CVPR 2025 SOTA')
else:
    print('RESULT: SPAI survives attack -> learned detectors are robust')

FileNotFoundError: No SPAI output CSV found for tag real

## 10. Save results figure + mount Drive

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Spectral Matching Attack: Real vs Fake vs Matched', fontsize=13, fontweight='bold')

# Radial profiles
prof_real = np.mean(real_profiles[:30], axis=0)
prof_fake = np.mean([radial_profile(load_gray(p)) for p in fake_paths[:30]], axis=0)
prof_matched = np.mean([radial_profile(load_gray(p)) for p in matched_paths[:30]], axis=0)

ax = axes[0]
ax.semilogy(prof_real,    color='blue',  lw=2, label='Real (RAISE-1k)')
ax.semilogy(prof_fake,    color='red',   lw=2, ls='--', label='Fake (SD1.4)')
ax.semilogy(prof_matched, color='green', lw=2, ls=':',  label='Matched')
ax.axvspan(0,  20,  alpha=0.07, color='green')
ax.axvspan(20, 60,  alpha=0.07, color='yellow')
ax.axvspan(60, 128, alpha=0.07, color='orange')
ax.set_title('Radial magnitude profile'); ax.set_xlabel('radius'); ax.legend()

# Band bar chart
bands = ['Low\n0-20', 'Mid\n20-60', 'High\n60+']
x = np.arange(3); w = 0.25
axes[1].bar(x-w, [spectral_report(load_gray(real_paths[0]))[k] for k in ('low','mid','high')], w, label='Real',    color='blue')
axes[1].bar(x,   [spectral_report(load_gray(fake_paths[0]))[k] for k in ('low','mid','high')], w, label='Fake',    color='red')
axes[1].bar(x+w, [spectral_report(load_gray(matched_paths[0]))[k] for k in ('low','mid','high')], w, label='Matched', color='green')
axes[1].set_yscale('log'); axes[1].set_xticks(x); axes[1].set_xticklabels(bands)
axes[1].set_title('Band energy'); axes[1].legend()

# SPAI score distribution
axes[2].hist(res_fake[score_col],    bins=20, alpha=0.6, color='red',   label='Fake (raw)')
axes[2].hist(res_matched[score_col], bins=20, alpha=0.6, color='green', label='Fake (matched)')
axes[2].hist(res_real[score_col],    bins=20, alpha=0.6, color='blue',  label='Real')
axes[2].axvline(0.5, color='black', ls='--', label='threshold')
axes[2].set_title('SPAI score distribution'); axes[2].set_xlabel('score (>0.5 = fake)'); axes[2].legend()

plt.tight_layout()
plt.savefig('/content/freqgen_result.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved /content/freqgen_result.png')

In [ ]:
import os, glob
DATA_DIR = '/content/data'
print("=== DISK STATE ===")
if os.path.exists(DATA_DIR):
    for root, dirs, files in os.walk(DATA_DIR):
        depth = root.replace(DATA_DIR,'').count(os.sep)
        if depth > 2: continue
        indent = '  '*depth
        print(f'{indent}{os.path.basename(root) or "data"}/  ({len(files)} files)')
else:
    print("DATA_DIR does not exist yet")

# Check Synthbuster download progress
tmp = '/tmp/synthbuster.zip'
if os.path.exists(tmp):
    sz = os.path.getsize(tmp)
    print(f"\nSynthbuster zip in progress: {sz/1e9:.2f} GB / 12 GB")
else:
    print("\nSynthbuster zip not in /tmp (either done extracting or not started)")

# Check COCO
coco = '/content/data/coco_val2017'
if os.path.exists(coco):
    n = len(os.listdir(coco))
    print(f"\nCOCO val2017: {n} images")
else:
    tmp_coco = '/tmp/coco_val.zip'
    if os.path.exists(tmp_coco):
        sz = os.path.getsize(tmp_coco)
        print(f"\nCOCO zip downloading: {sz/1e6:.0f} MB / 778 MB")
    else:
        print("\nCOCO: not started")